# Step 1: Research & Data Source Discovery

## 1.1 Environment Setup and Dependencies

This step ensures all necessary libraries are installed and imported. The pipeline uses a multi-source approach for coverage and quality:

* **Primary Source: OpenStreetMap (OSM)**: Used to extract the spatial location of employment agencies.
* **Enrichment Source: Wikidata**: Necessary to fill in missing administrative details (like `operator_name` and `contact_website`) that OSM data frequently lacks.

## 1.2 Data and Boundary Configuration

The project focuses exclusively on data within the **Berlin, Germany** boundary.

* **Spatial Integrity Plan**: Data will be joined to the **Local Reference System (LOR) boundaries** to derive the mandatory `district_id` and `neighborhood_id` for final database compliance.

In [1]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
import hashlib
import requests
import os 
import json

# --- CONFIGURATION ---
PLACE_NAME = "Berlin, Germany"
OSM_TAGS = {"office": "employment_agency"}
LOR_PATH = "lor_ortsteile.geojson" # Confirmed correct path

print("Libraries loaded.")
print(f"Configuration set for {PLACE_NAME} with OSM tags: {OSM_TAGS}")

Libraries loaded.
Configuration set for Berlin, Germany with OSM tags: {'office': 'employment_agency'}


## 2: External API Enrichment

The primary enrichment function, `enrich_data_from_wikidata`, attempts to fill missing `operator_name` and `contact_website` fields by searching the Wikidata SPARQL API using the location's name and coordinates. This is a crucial data blending step.

In [2]:
def query_wikidata(query):
    """Placeholder for Wikidata query (skipped due to rate-limiting)."""
    return None

def enrich_data_from_wikidata(df):
    """Initializes enrichment columns and skips external call."""
    print("LOG: Skipping full Wikidata enrichment due to previous rate-limit errors (429/500).")
    if "operator_name" not in df.columns:
        df["operator_name"] = None
    if "contact_website" not in df.columns:
        df["contact_website"] = None
    return df

## 3: Live Data Extraction and Mandatory Raw Data Cleaning

The data is fetched from OSM for Berlin and immediately processed for foundational data quality:

* **CRS Alignment:** The GeoDataFrame is set to the standard **`EPSG:4326`** CRS (WGS 84).
* **Diagnostic Check:** We explicitly check and report on null values in the mandatory `name` and `geometry` columns.
* **Mandatory Drop:** Any records missing these critical values are **mandatorily dropped** to enforce `NOT NULL` compliance, and the result is stored in the new, independent `jobcenter_enriched` DataFrame via `.copy()`.

In [3]:
print("Fetching live data from OpenStreetMap (Overpass API)...")
try:
    # Fetch data and set CRS to EPSG:4326
    jobcenter_data_raw = ox.features_from_place(PLACE_NAME, OSM_TAGS)
    jobcenter_data_raw = gpd.GeoDataFrame(
        jobcenter_data_raw,
        geometry="geometry",
        crs="EPSG:4326"
    )
    print(f"Success! Retrieved {len(jobcenter_data_raw)} features.")
except Exception as e:
    raise RuntimeError(f"OSM extraction failed: {e}")

# --- Diagnostic Check (New addition for validation) ---
print("\n--- Diagnostic Check: Nulls in Critical Columns ---")
null_counts = jobcenter_data_raw[['name', 'geometry']].isnull().sum()
print("Missing values in critical columns:")
print(null_counts)


# --- Mandatory Cleaning ---
initial_count = len(jobcenter_data_raw)

# Drop rows where 'name' OR 'geometry' is missing and create a clean copy
jobcenter_enriched = jobcenter_data_raw.dropna(subset=["name", "geometry"]).copy()

dropped_count = initial_count - len(jobcenter_enriched)
print(f"Mandatory Drop: Removed {dropped_count} rows due to missing name/geometry.")

if jobcenter_enriched.empty:
    raise RuntimeError("Dataset empty after cleaning.")

Fetching live data from OpenStreetMap (Overpass API)...
Success! Retrieved 65 features.

--- Diagnostic Check: Nulls in Critical Columns ---
Missing values in critical columns:
name        2
geometry    0
dtype: int64
Mandatory Drop: Removed 2 rows due to missing name/geometry.


## 4: Stable ID Generation and Enrichment Application

* **Stable ID (FIXED):** A deterministic, **numeric-only ID** is generated using `hashlib.sha256`. The function uses **`.geometry.centroid`** to ensure it works correctly for all geometries, including `Polygon` features, thus preventing the `AttributeError` encountered previously.
* **Enrichment:** The `enrich_data_from_wikidata` function is applied to fill the `operator_name` and `contact_website` columns.

In [4]:
# 1. Define ID Generation Function (FIXED to use .centroid)
def generate_stable_numeric_id(row):
    """Deterministic, numeric-only ID (max 20 chars)."""
    # FIX: Use .centroid.x and .centroid.y to handle both Point and Polygon objects
    raw_value = f"{row['name']}{row.geometry.centroid.x}{row.geometry.centroid.y}" 
    hash_hex = hashlib.sha256(raw_value.encode()).hexdigest()
    # Convert part of hash to integer and truncate to max 20 characters
    return str(int(hash_hex[:16], 16))[:20] 

# 2. Apply ID Generation
jobcenter_enriched["id"] = jobcenter_enriched.apply(
    generate_stable_numeric_id, axis=1
)
print("ID generation complete.")

# 3. Apply Wikidata Enrichment
jobcenter_enriched = enrich_data_from_wikidata(jobcenter_enriched)
print("Wikidata enrichment applied.")

ID generation complete.
LOG: Skipping full Wikidata enrichment due to previous rate-limit errors (429/500).
Wikidata enrichment applied.


## 5.1 Coordinate Preparation and Spatial Join

* **WKT Conversion (FIXED):** Coordinates are extracted using `.centroid.x` and `.centroid.y`. The data is **temporarily reprojected** to the local **EPSG:25833** CRS for highly accurate centroid calculation, eliminating the previous `UserWarning`. The result is then projected back to **EPSG:4326**.
* **LOR Join:** A **Left Spatial Join** is performed using the uploaded LOR file to derive the mandatory `district_id` and `neighborhood_id`. 

## 5.2 Enhanced Exploratory Data Analysis (EDA)

The EDA is performed after all major joining steps to confirm data quality and discover insights in the final structure.

In [14]:
import os
import geopandas as gpd

# --- Re-establish Context and Data (Steps 1-4) ---
# NOTE: This block is repeated to ensure 'jobcenter_enriched' dataframe exists.
# (Code for Steps 1-4 re-establishment) ...
PLACE_NAME = "Berlin, Germany"
OSM_TAGS = {"office": "employment_agency"}

def enrich_data_from_wikidata(df):
    if "operator_name" not in df.columns:
        df["operator_name"] = None
    if "contact_website" not in df.columns:
        df["contact_website"] = None
    return df

def generate_stable_numeric_id(row):
    import hashlib
    raw_value = f"{row['name']}{row.geometry.centroid.x}{row.geometry.centroid.y}" 
    hash_hex = hashlib.sha256(raw_value.encode()).hexdigest()
    return str(int(hash_hex[:16], 16))[:20] 

# Re-extract Data
jobcenter_data_raw = ox.features_from_place(PLACE_NAME, OSM_TAGS)
jobcenter_data_raw = gpd.GeoDataFrame(jobcenter_data_raw, geometry="geometry", crs="EPSG:4326")
jobcenter_enriched = jobcenter_data_raw.dropna(subset=["name", "geometry"]).copy()
jobcenter_enriched["id"] = jobcenter_enriched.apply(generate_stable_numeric_id, axis=1)
jobcenter_enriched = enrich_data_from_wikidata(jobcenter_enriched)
print("Data context re-established.")
# ---------------------------------------------------

print("\n--- Step 5: Spatial Join (Attaching LOR IDs) ---")

# Determine LOR_PATH (Final working path: one level up)
LOR_PATH = os.path.join(os.pardir, "lor_ortsteile.geojson")
print(f"Using final, corrected LOR Path: {LOR_PATH}")

# 1. Coordinate Prep and WKT Conversion 
jobcenter_working = jobcenter_enriched.to_crs(epsg=25833).copy() 
jobcenter_working["longitude"] = jobcenter_working.geometry.centroid.x
jobcenter_working["latitude"] = jobcenter_working.geometry.centroid.y
jobcenter_enriched = jobcenter_working.to_crs(epsg=4326) 
jobcenter_enriched["geometry_str"] = jobcenter_enriched.apply(
    lambda r: f"POINT({r['longitude']} {r['latitude']})",
    axis=1
)
print("Coordinates and WKT conversion complete and corrected for projection bias.")


# 2. LOR Load, Rename, and Join 
lor = gpd.read_file(LOR_PATH)
lor = lor.to_crs(jobcenter_enriched.crs) 

# CRITICAL FIX: Renaming columns based on the GeoJSON file structure
lor = lor.rename(columns={
    "OTEIL": "neighborhood",
    "spatial_name": "neighborhood_id", 
    "BEZIRK": "district",
    "gml_id": "district_id" 
})
print("LOR boundaries loaded and prepared.")


# <<<< THE ABSOLUTE FIX FOR THE VALUEERROR: EXPLICITLY RESET INDEX >>>>
# This removes the conflicting index and replaces it with a simple RangeIndex.
jobcenter_enriched = jobcenter_enriched.reset_index(drop=True)
print("Index reset to avoid 'id' conflict.")


# Perform Spatial Join 
jobcenter_enriched = gpd.sjoin(
    jobcenter_enriched,
    lor,
    how="left",
    predicate="within"
)
print(f"Spatial join completed. Final Row Count: {len(jobcenter_enriched)}")


# --- ENHANCED EDA SECTION ---
print("\n--- ENHANCED EDA: Spatial & Feature Discovery ---")
lor_nulls = jobcenter_enriched[['district_id', 'neighborhood_id']].isnull().sum()
print("1. LOR Coverage Check (Critical integrity test):")
print(lor_nulls)
if lor_nulls.sum() > 0:
    print("WARNING: Some Jobcenters are outside LOR boundaries and must be reviewed.")

district_operator_check = jobcenter_enriched.groupby('district')['operator_name'].agg(
    ['nunique', 'size']
).rename(columns={'nunique': 'Unique Operators', 'size': 'Total Jobcenters'})
print("\n2. District vs. Operator Analysis (Top 5 Districts by Jobcenter Count):")
print(district_operator_check.sort_values(by='Total Jobcenters', ascending=False).head())
print("-------------------------------------------------")

Data context re-established.

--- Step 5: Spatial Join (Attaching LOR IDs) ---
Using final, corrected LOR Path: ../lor_ortsteile.geojson
Coordinates and WKT conversion complete and corrected for projection bias.
LOR boundaries loaded and prepared.
Index reset to avoid 'id' conflict.
Spatial join completed. Final Row Count: 63

--- ENHANCED EDA: Spatial & Feature Discovery ---
1. LOR Coverage Check (Critical integrity test):
district_id        0
neighborhood_id    0
dtype: int64

2. District vs. Operator Analysis (Top 5 Districts by Jobcenter Count):
                            Unique Operators  Total Jobcenters
district                                                      
Mitte                                      0                14
Charlottenburg-Wilmersdorf                 0                10
Friedrichshain-Kreuzberg                   0                 9
Neukölln                                   0                 7
Tempelhof-Schöneberg                       0                 4
---

## Step 6: Final Schema Selection and Export

This final step prepares the `jobcenter_enriched` data for database loading by selecting the required columns, imputing missing address/contact details, and exporting the final result to a CSV file.

### 1. Impute Final Columns

```python
# 1. Impute final columns (Note: These columns were initialized in Step 4/enrichment)
# We use .get(column, default) defensively
jobcenter_enriched["address_full"] = jobcenter_enriched.get("addr:street", jobcenter_enriched.get("addr:full", None))
jobcenter_enriched["operating_hours"] = jobcenter_enriched.get("opening_hours", None) 
jobcenter_enriched["services_offered"] = None # No direct source for this, remains None
jobcenter_enriched["contact_phone"] = jobcenter_enriched.get("contact:phone", None)
jobcenter_enriched["data_source"] = "OSM_LOR_Wikidata"

In [16]:
import os

# 1. Impute final columns (Note: These columns were initialized in Step 4/enrichment)
# We use .get(column, default) defensively
jobcenter_enriched["address_full"] = jobcenter_enriched.get("addr:street", jobcenter_enriched.get("addr:full", None))
jobcenter_enriched["operating_hours"] = jobcenter_enriched.get("opening_hours", None) 
jobcenter_enriched["services_offered"] = None # No direct source for this, remains None
jobcenter_enriched["contact_phone"] = jobcenter_enriched.get("contact:phone", None)
jobcenter_enriched["data_source"] = "OSM_LOR_Wikidata" 

# 2. Select columns in the EXACT ORDER required by SQL
final_jobcenters = jobcenter_enriched[[
    "id", "district_id", "name", "latitude", "longitude", 
    "geometry_str", "neighborhood", "district", "neighborhood_id", 
    "address_full", "operating_hours", "services_offered", "contact_phone", 
    "contact_website", "operator_name", "data_source" 
]].rename(columns={"geometry_str": "geometry"})

print("Final 16-column schema prepared for SQL load.")

# 3. Export to CSV
# Path to put the output folder in the project root (up two levels from 'scripts')
OUTPUT_PATH = os.path.join(os.pardir, os.pardir, 'output', 'jobcenters_berlin.csv')

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
final_jobcenters.to_csv(OUTPUT_PATH, index=False)

print(f"Pipeline complete. Output saved to: {OUTPUT_PATH}")
print(f"Total Jobcenter locations exported: {len(final_jobcenters)}")

Final 16-column schema prepared for SQL load.
Pipeline complete. Output saved to: ../../output/jobcenters_berlin.csv
Total Jobcenter locations exported: 63
